Ready the Chatbot

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# ---------------- Configuration ----------------

model_id = "Qwen/Qwen2-0.5B-Instruct"

# ---------------- 1. Load Model & Tokenizer ----------------

print(f"Loading model: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

# ---------------- 2. Create Pipeline ----------------

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.7,
)

# ---------------- 3. LangChain Wrapper ----------------

raw_llm = HuggingFacePipeline(pipeline=pipe)

# ---------------- 4. Prompt Formatter ----------------

def format_for_qwen(input_dict):
    """
    input_dict expects:
    {
        "history": [...],
        "input": "user message"
    }
    """

    history = input_dict.get("history", [])
    user_input = input_dict.get("input", "")

    # Construct the messages list
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        }
    ]

    messages.extend(history)

    messages.append(
        {
            "role": "user",
            "content": user_input
        }
    )

    # Apply the specific chat template for Qwen
    # tokenize=False returns a string, which is what HuggingFacePipeline needs
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

print("Setup complete")

d:\AI-Course(DSTP3.0-BATCH-03)\Week11\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: Qwen/Qwen2-0.5B-Instruct...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2307.21it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Setup complete


Create Flask App

In [3]:
# import libraries
import threading
import time
import requests
from flask import Flask, request, jsonify
from flask_cors import CORS
import logging

In [4]:
if 'raw_llm' not in globals() or 'tokenizer' not in globals():
    raise EnvironmentError(
        "⚠️ ERROR: 'raw_llm' or 'tokenizer' not found. Please run the Model Setup cell first."
    )

print("✅ Environment check passed. LLM found.")

✅ Environment check passed. LLM found.


In [5]:
# ---------------------------------------------------
# 1. Helper Functions
# ---------------------------------------------------

def create_qwen_prompt(user_text, history=None):
    if history is None:
        history = []

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        }
    ]

    messages.extend(history)

    messages.append(
        {
            "role": "user",
            "content": user_text
        }
    )

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def clean_qwen_response(response: str, prompt: str):
    if prompt in response:
        response = response.replace(prompt, "").strip()

    if "<|im_start|>assistant" in response:
        return response.split("<|im_start|>assistant")[-1].strip()

    return response.strip()

In [6]:
# ---------------------------------------------------
# 2. Flask App Setup
# ---------------------------------------------------

app = Flask(__name__)

CORS(app)   # Cross origin resource sharing

log = logging.getLogger("werkzeug")
log.setLevel(logging.ERROR)

In [7]:
# ---------------------------------------------------
# 3. Routes
# ---------------------------------------------------

@app.route("/chat", methods=["POST"])
def chat():
    try:
        data = request.json

        formatted_prompt = create_qwen_prompt(
            data.get("message", "")
        )

        raw_response = raw_llm.invoke(formatted_prompt)

        ai_response = clean_qwen_response(
            raw_response,
            formatted_prompt
        )

        return jsonify(
            {
                "status": "success",
                "response": ai_response
            }
        )

    except Exception as e:
        return jsonify({"error": str(e)}), 500

In [8]:
# ---------------------------------------------------
# 4. Start Server
# ---------------------------------------------------

def run_flask():
    print("🚀 Flask Server started on http://127.0.0.1:5000")
    app.run(
        port=5000,
        use_reloader=False
    )


t = threading.Thread(target=run_flask)
t.daemon = True
t.start()

print("⏳ Waiting for server to be ready...")
time.sleep(3)

🚀 Flask Server started on http://127.0.0.1:5000⏳ Waiting for server to be ready...

 * Serving Flask app '__main__'
 * Debug mode: off


In [9]:
# ---------------------------------------------------
# TEST the General Chat
# ---------------------------------------------------

print("\n[TEST] Endpoint: /chatbot")
print("-" * 40)

try:
    resp = requests.post(
        "http://127.0.0.1:5000/chat",
        json={
            "message": "Introduce yourself briefly."
        }
    )

    if resp.status_code == 200:
        print("User: Introduce yourself briefly.")
        print(f"AI: {resp.json()['response']}")
    else:
        print(f"Error: {resp.status_code}")

except Exception as e:
    print(f"Failed: {e}")

time.sleep(1)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[TEST] Endpoint: /chatbot
----------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


User: Introduce yourself briefly.
AI: I am an artificial intelligence designed to assist with various tasks, such as answering questions, generating text, providing information, and much more. I can communicate in multiple languages and interact with users through natural language processing techniques. My purpose is to help people find the information they need, solve problems, and engage in meaningful conversations.
